In [6]:
import os
import fiftyone as fo

# 1. Define paths
IMAGE_DIR = "./dataset/crop"
LABELS_DIR = "./dataset/labels"
CLASSES = ["cat"]
dataset_name = "cat-playground-dataset"

# 2. Clear old database instances if you ran it before
if fo.dataset_exists(dataset_name):
    fo.delete_dataset(dataset_name)

print("Creating FiftyOne dataset instance...")
dataset = fo.Dataset(name=dataset_name)
samples = []

# --- THE FIX: Sort the filenames alphabetically so they match capture order ---
sorted_img_names = sorted(os.listdir(IMAGE_DIR))

# 3. Match raw images with your new YOLO text files in order
for img_name in sorted_img_names:
    # Only process valid image files
    if not img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
        continue
        
    img_path = os.path.join(IMAGE_DIR, img_name)
    sample = fo.Sample(filepath=img_path)
    
    # Locate the matching text coordinate file
    base_name = os.path.splitext(img_name)[0]
    label_path = os.path.join(LABELS_DIR, f"{base_name}.txt")
    
    if os.path.exists(label_path):
        detections = []
        with open(label_path, "r") as f:
            lines = f.readlines()
            
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id, x_center, y_center, width, height = map(float, parts)
                
                # YOLO format is center-relative: [x_center, y_center, width, height]
                # FiftyOne requires top-left bounding boxes: [xmin, ymin, width, height]
                xmin = x_center - (width / 2)
                ymin = y_center - (height / 2)
                
                detections.append(
                    fo.Detection(
                        label=CLASSES[int(class_id)],
                        bounding_box=[xmin, ymin, width, height]
                    )
                )
        if detections:
            sample["ground_truth"] = fo.Detections(detections=detections)
            
    samples.append(sample)

# 4. Save to database and start the local server
dataset.add_samples(samples)
print(f"Successfully loaded {len(dataset)} samples chronologically into FiftyOne.")


Creating FiftyOne dataset instance...
 100% |███████████████| 6197/6197 [848.3ms elapsed, 0s remaining, 7.3K samples/s]      
Successfully loaded 6197 samples chronologically into FiftyOne.


In [ ]:
import os
import fiftyone as fo

dataset = fo.load_dataset("cat-playground-dataset")

tracked_filepaths = set(dataset.values("filepath"))

IMAGE_DIR = "./dataset/crop"
LABELS_DIR = "./dataset/labels"

images_deleted = 0
labels_deleted = 0

print("Scanning for unlabelled/untracked files on disk...")

for img_name in os.listdir(IMAGE_DIR):
    if not img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
        continue
        
    local_img_path = os.path.abspath(os.path.join(IMAGE_DIR, img_name))
    
    if local_img_path not in tracked_filepaths:
        if os.path.exists(local_img_path):
            os.remove(local_img_path)
            images_deleted += 1
            
        base_name = os.path.splitext(img_name)[0]
        local_label_path = os.path.join(LABELS_DIR, f"{base_name}.txt")
        if os.path.exists(local_label_path):
            os.remove(local_label_path)
            labels_deleted += 1

print(f"🔥 Disk Synchronization Complete!")
print(f"Successfully deleted {images_deleted} unlabelled images and {labels_deleted} matching labels from your Mac.")



Scanning for unlabelled/untracked files on disk...
🔥 Disk Synchronization Complete!
Successfully deleted 293 unlabelled images and 293 matching labels from your Mac.


In [10]:
import os
import fiftyone as fo

# 1. Target your exact custom tag group
target_tag = "persons and repetitive phots"
view_to_delete = dataset.match_tags(target_tag)

print(f"Safety Verification: Found {len(view_to_delete)} samples tagged with '{target_tag}'.")

# Guardrail to make sure we are operating on the correct chunk
if len(view_to_delete) == 1626:
    IMAGE_DIR = "./dataset/crop"
    LABELS_DIR = "./dataset/labels"
    
    images_deleted = 0
    labels_deleted = 0
    
    # 2. Iterate and physically wipe files from local storage
    for sample in view_to_delete:
        local_img_path = os.path.abspath(sample.filepath)
        
        # Erase the raw image frame
        if os.path.exists(local_img_path):
            os.remove(local_img_path)
            images_deleted += 1
            
        # Erase the matching YOLO text file containing the bounding box coordinates
        base_name = os.path.splitext(os.path.basename(local_img_path))[0]
        local_label_path = os.path.join(LABELS_DIR, f"{base_name}.txt")
        if os.path.exists(local_label_path):
            os.remove(local_label_path)
            labels_deleted += 1
            
    # 3. Clean the records out of FiftyOne's app session database
    dataset.delete_samples(view_to_delete)
    
    print(f"🔥 Success! Physically removed {images_deleted} images and {labels_deleted} labels from your Mac.")
else:
    print(f"Execution halted: The database returned {len(view_to_delete)} samples for that tag instead of 1626.")
    print("Please make sure you don't have an active filter checked on your FiftyOne sidebar screen!")

Safety Verification: Found 1626 samples tagged with 'persons and repetitive phots'.
🔥 Success! Physically removed 1626 images and 1626 labels from your Mac.
